# Datamine MGSORT Process Example

This notebook demonstrates how to configure and run the **`mgsort`** process wrapper in `dmstudio`.

## Process Description

## MGSORT

Sort a Datamine (.dm, .dmx) file into ascending or descending order of keyfields. Ascending order is the default; descending order is specified by setting the optional parameter @**ORDER** =2.

At least one keyfield must be specified and must appear as an explicit field. For alphanumeric fields, the collating sequence is the standard Datamine-specific sort sequence for text. The fields are sorted in the order in which they are specified. If a specified keyfield is not in the file, it is ignored. A warning message is given if @**PRINT** >=1.

**Note** : SORTX and MGSORT are identical, although separate process names have been retained and **SORTX** still accessible for legacy macro purposes.

Where sufficient memory exists, the table is sorted entirely within memory. However, if the memory required exceeds that available, the table is sorted in separate 'chunks', each temporarily stored to disk and then merged together for the final write. The maximum number of records that can be sorted is therefore limited only by available disk space.

This process includes the KEYSFRST option which, by default, maintains legacy behavior by reordering key fields in the output file so that KEY1 becomes the first field, KEY2 the second field, and so on. If you specify that key fields are not reordered, then the order of fields in the OUT and IN file will be identical.

This process allows you to determine how the sort is performed with regards to duplicate key field values, using the **ROWORDER** parameter.

* **Note** (Although the order of fields in a file does not affect subsequent processing, it makes it more difficult to review the file using Datamine's Table Editor. For example, a borehole file which is sorted by BHID, FROM and TO is more difficult to manually validate if these fields are not adjacent.):

### Input Files:

* **in** (Table):
  File to be sorted.
  Required=Yes

### Output Files:

* **out** (Table):
  Sorted file.
  Required=Yes

### Fields:

* **keys** (Undefined : Undefined):
  Keyfields for sorting .
  Default=Undefined
  Required=No

### Parameters:

* **order**:

* **Options** (1: For ascending order; 2: For descending order (1).):
  Range=1,2
  Values=1,2
  Default=1
  Required=No

* **keysfrst**:

* **Options** (0: output fields in the same order as the input table; 1: output key fields):
  first
  Range=0,1
  Values=0,1
  Default=1
  Required=No

* **roworder**:

* **Options** (0: Rows which contain duplicate key field values could be in any order):
  (faster); 1: Rows which contain duplicate key field values will be in the input file
  order (slower) (1)
  Range=0,1
  Values=0,1
  Default=1
  Required=No

* **keytol**:
  **KEYTOL** is the tolerance value used to test whether numeric key values are equal. It
  must be greater than or equal to zero. It replaces the previous heuristic comparison
  method. If **KEYTOL** is set to a negative value then zero is used. In a macro
  **KEYTOL** can be set to absent using -. "@**KEYTOL** =-" This will revert to legacy
  behaviour and use a heuristic comparison in relational commands and zero in sort.
  Range=0,+
  Values=Undefined
  Default=0.00001
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Verify Active Project
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Verify that the active project matches this folder (case-insensitive) to prevent writing files to the wrong place
active_folder = os.path.normpath(oScript.ActiveProject.Folder).lower()
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()

if active_folder != notebook_folder:
    raise RuntimeError(
        f"Active Datamine Project ({active_folder}) does not match this notebook's folder ({notebook_folder}).\n"
        "Please open the 'Project.rmproj' in this folder inside Datamine Studio RM first!"
    )
print("Active project verified successfully.")

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('mgsort')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine help database locally to this folder using a `t_` prefix. Paths are resolved relatively to ensure portability.

In [ ]:
# Step 3: Initialize example project dataset using relative paths
# Resolve relative path to repository's help database dynamically
repo_root = os.path.abspath(os.path.join(notebook_folder, '..', '..', '..'))
help_db = os.path.join(repo_root, 'datamine_help', 'Database', 'DMTutorials', 'Data', 'VBOP', 'Datamine')

files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

for filename in files_to_copy:
    src = os.path.join(help_db, filename)
    # Strip _vb_ prefix and prepend t_ for local usage
    local_name = "t_" + filename.replace("_vb_", "")
    dst = os.path.join(notebook_folder, local_name)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Initialized dataset: {local_name}")
    else:
        print(f"Warning: Source {filename} not found in help database.")

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute mgsort
print("Running mgsort...")
dm_cmd.mgsort(
    in_i='t_assays',  # required
    out_o='t_mgsort_out',  # required
    # keys_f=['BHID'],  # optional
    # order_p=1,  # optional
    # keysfrst_p=1,  # optional
    # roworder_p=1,  # optional
    # keytol_p=1e-05,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("mgsort execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = os.path.join(notebook_folder, 't_mgsort_out.dmx')
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob(os.path.join(notebook_folder, "t_*.*"))
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    path = os.path.join(notebook_folder, f)
    if os.path.exists(path):
        try:
            os.remove(path)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = os.path.join(notebook_folder, '__pycache__')
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")